## 0. Set-up

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch, torchvision

import torch.nn as nn
import torch.optim as optim

from utils import *
from data_loader import data_loader
import models.CNN as cnn_model
import models.CNN2 as cnn_model2

In [ ]:
if torch.cuda.is_available() :
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built() :
    DEVICE = torch.device("mps")
else :
    DEVICE = torch.device("cpu")
print(f"Using device on [{DEVICE}]")

<u>**Note:**</u>
Since we are studying temporal corruption between different optimizers, we need to be able to reproduce the same corruption for each of them. Typically, the order of the data, which is known to impact performance, should always be the same. Thus, we set a seed that will be used throughout our work.

In [ ]:
torch.manual_seed(42)
if torch.cuda.is_available() :
    torch.cuda.manual_seed_all(42)

## 1. Loading the data

In [ ]:
train_loader, test_loader, classes = data_loader(batchsize = 128)
print(f"\nLabels of the dataset are {classes}.")

data_iterator = iter(train_loader)

In [ ]:
images, labels = next(data_iterator)
show_image(images[0])

## 2. Basic CNN model

<u>Source:</u> [PyTorch documentation](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html#)

In [ ]:
# Loading the convolutional Neural Network
net = cnn_model.Net()

In [ ]:
# We define a loss function and an optimizer
# SGD optimizer for now with by-default parameters
criterion = nn.CrossEntropyLoss()
optimizer_SGD = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
# we train the network 

nb_epochs = 2

"""for epoch in range(nb_epochs) :

    running_loss = 0.0

    for i, data in enumerate(train_loader, 0) :
        inputs, labels = data

        optimizer_SGD.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_SGD.step()

        running_loss += loss.item()
        if i % 2000 == 1999 :
            print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}")
            running_loss = 0.0
    
print("Finished training")"""

In [ ]:
# we save our trained model!!!!
PATH = "./trained_models/cifar_net_2epochs.pth"
# torch.save(net.state_dict(), PATH)

In [ ]:
# if model already trained, run the following
net = cnn_model.Net()
net.load_state_dict(torch.load(PATH, weights_only = True))

In [ ]:
# We test the network on the test data
# We compare the neural network outputs with the ground truth
test_data_iterator = iter(test_loader)
images, labels = next(test_data_iterator)

# show_image(torchvision.utils.make_grid(images))
print("Ground Truth:", " ".join(f"{classes[labels[j]]:5s}" for j in range(4)))

outputs = net(images)
_, prediction = torch.max(outputs, 1)
print("Predicted: ", " ".join(f"{classes[prediction[j]]:5s}" for j in range(4)))

In [ ]:
# performance on the entire dataset
correct, total = 0, 0

with torch.no_grad() :
    for data in test_loader :
        images, labels = data
        outputs = net(images)
        _, predictions = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

print(f"Accuracy of the network trained on {nb_epochs} epochs on the 10000 test images: {100 * correct / total} %.")

## 3. Bigger CNN model

<u>Source:</u> [*Convolutional Neural Network (CNN) for Image Detection and Recognition*, by Chauban et al](https://ieeexplore.ieee.org/abstract/document/8703316)

We refer to this paper and use their model's architecture for a convolutional network that they trained on CIFAR10, obtaining an accuracy of 80.01% on the test dataset (on CPU).

In [ ]:
net2 = cnn_model2.CNN_for_CIFAR()

In [ ]:
# We define a loss function and an optimizer
# SGD optimizer for now with by-default parameters
criterion = nn.CrossEntropyLoss()
optimizer_SGD = optim.SGD(net2.parameters(), lr=0.001, momentum=0.9)

In [ ]:
# we train the network 

nb_epochs = 1

for epoch in range(nb_epochs) :

    running_loss = 0.0

    for i, data in enumerate(train_loader, 0) :
        inputs, labels = data

        optimizer_SGD.zero_grad()

        # forward + backward + optimize
        outputs = net2(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_SGD.step()

        running_loss += loss.item()
        if i % 2000 == 1999 :
            print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}")
            running_loss = 0.0
    
print("Finished training")

In [ ]:
PATH = "./trained_models/cifar_net_advanced_1epoch.pth"
torch.save(net2.state_dict(), PATH)

In [ ]:
# We test the network on the test data
# We compare the neural network outputs with the ground truth
test_data_iterator = iter(test_loader)
images, labels = next(test_data_iterator)

# show_image(torchvision.utils.make_grid(images))
print("Ground Truth:", " ".join(f"{classes[labels[j]]:5s}" for j in range(4)))

outputs = net2(images)
_, prediction = torch.max(outputs, 1)
print("Predicted: ", " ".join(f"{classes[prediction[j]]:5s}" for j in range(4)))

In [ ]:
# performance on the entire dataset
correct, total = 0, 0

with torch.no_grad() :
    for data in test_loader :
        images, labels = data
        outputs = net2(images)
        _, predictions = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

print(f"Accuracy of the network trained on {nb_epochs} epochs on the 10000 test images: {100 * correct / total} %.")